# Comparative study: evaluation of LDA, NMF and BERTopic (1955–1970)

In [ ]:
# please install this version scipy==1.12.0, gensim==4.2.0, top2vec==1.0.34,bertopic==0.16.2

In [ ]:
import sys


sys.path.append('../evaluation')

In [ ]:
%cd ../..

In [ ]:
import numpy as np
import pandas as pd
import pickle

from umap import UMAP
from hdbscan import HDBSCAN

In [ ]:
# The below Top2vec is custom model - modified the original code for our comparion study
from Top2Vec import Top2Vec_custom

from sentence_transformers import SentenceTransformer

In [ ]:
from evaluation import Trainer
from data import DataLoader
from datetime import datetime
from LDAtfidf import LDA_Tfidf
from NMFtfidf import NMF_tfidf
from LDA_NER import LDA_NER

# LDA-tfidf

In [ ]:
for i, random_state in enumerate([0, 21, 42]):
    dataset, custom = "../../datasets/compare_study_dataset/word_embedding_data", True
    params = {"num_topics": [(i+1)*10 for i in range(5)],"random_state": random_state}

    trainer = Trainer(dataset=dataset,
                      model_name="ldatfidf",
                      params=params,
                      custom_model='LDA_Tfidf',
                      custom_dataset=custom,
                      verbose=True)
    results_LDA,output_LDA = trainer.train(save=f"../results/lowercased/LDA_Tfidf/LDA_result_{i+1}")
    with open (f'../results/lowercased/LDA_Tfidf/LDA_50_TOPICS_{i+1}.pkl', 'wb') as file:
        pickle.dump(output_LDA,file)

In [ ]:
output_lda_df=pd.DataFrame(output_LDA['topics'])

# LDA-ner

In [ ]:
for i, random_state in enumerate([42]):
    dataset, custom = "../../datasets/compare_study_dataset/word_embedding_data", True
    params = {"num_topics": [(i+1)*10 for i in range(5)],"random_state": random_state}

    trainer = Trainer(dataset=dataset,
                      model_name="LDA_NER",
                      params=params,
                      custom_model='LDA_NER',
                      custom_dataset=custom,
                      verbose=True)
    results_LDA_NER,output_LDA_NER = trainer.train(save=f"../evaluation_results/Results_Comp_Study/1955_1970/results/lowercased/LDA_NER/LDA_result_{i+1}")
    with open (f'../evaluation_results/Results_Comp_Study/1955_1970/results/lowercased/LDA_NER/LDA_50_TOPICS_{i+1}.pkl', 'wb') as file:
        pickle.dump(output_LDA,file)

In [ ]:
with open (f'../evaluation_results/Results_Comp_Study/1955_1970/results/lowercased/LDA_NER/LDA_50_TOPICS_{i+1}.pkl', 'wb') as file:
        pickle.dump(output_LDA_NER,file)

In [ ]:
LDA_NER_topics_df = pd.DataFrame(output_LDA_NER['topics'])

# NMF-tfidf

In [ ]:
import pickle

for i, random_state in enumerate([0, 21, 42]):
    dataset, custom = "../../datasets/compare_study_dataset/word_embedding_data/", True
    params = {"num_topics": [(i+1)*10 for i in range(5)], "random_state": random_state}

    trainer = Trainer(dataset=dataset,
                      model_name="NMFtfidf",
                      params=params,
                      custom_dataset=custom,
                      custom_model='NMF_tfidf',
                      verbose=True)
    results,output = trainer.train(save=f"../results/lowercased/NMF_tfidf/NMF_result_{i+1}")
    with open (f'../results/lowercased/NMF_tfidf/NMF_50_TOPICS_{i+1}.pkl', 'wb') as file:
        pickle.dump(output,file)

In [ ]:
output_NMF= pd.DataFrame(output['topics'])

# NMF-ner

In [ ]:
import pickle

for i, random_state in enumerate([42]):
    dataset, custom = "../../datasets/compare_study_dataset/word_embedding_data/", True
    params = {"num_topics": [(i+1)*10 for i in range(5)], "random_state": random_state}

    trainer = Trainer(dataset=dataset,
                      model_name="NMF_NER",
                      params=params,
                      custom_dataset=custom,
                      custom_model='NMF_NER',
                      verbose=True)
    results_nmf_ner,output_nmf_ner = trainer.train(save=f"../evaluation_results/Results_Comp_Study/1955_1970/results/lowercased/NMF_NER/NMF_result_{i+1}")
    with open (f'../evaluation_results/Results_Comp_Study/1955_1970/results/lowercased/NMF_NER/NMF_50_TOPICS_{i+1}.pkl', 'wb') as file:
        pickle.dump(output,file)

In [ ]:
with open (f'../evaluation_results/Results_Comp_Study/1955_1970/results/lowercased/NMF_NER/NMF_50_TOPICS_{i+1}.pkl', 'wb') as file:
      pickle.dump(output_nmf_ner,file)

# BERTopic

### Lightly cleaned text (sentence-level)

In [ ]:
sent_embeddings_mpnet_aggregated=np.load("../../datasets/compare_study_dataset/1955_1970/aggregation/sentence_embeddings_octis_MPnet.npy")
sent_embeddings_distilroberta_aggregated=np.load("../../datasets/compare_study_dataset/1955_1970/aggregation/sentence_embeddings_octis_distilroberta.npy")
sent_embeddings_alibaba_aggregated=np.load("../../datasets/compare_study_dataset/1955_1970/aggregation/sentence_embeddings_octis_Alibaba-NLP_gte-base-en-v1.5.npy")

sent_embeddings_jina_aggregated=np.load("../../datasets/compare_study_dataset/1955_1970/aggregation/sentence_embeddings_octis_jina_v2_base_en.npy")

### MPNet (all-mpnet-base-v2)

In [ ]:
# DEFAULT SETTING with aggregation

In [ ]:
from sentence_transformers import SentenceTransformer

# Prepare data
dataset, custom = "../../datasets/compare_study_dataset/1955_1970/aggregation/sentence_embedding_data_384", True
data_loader = DataLoader(dataset)
_, timestamps = data_loader.load_docs()
data = data_loader.load_octis(custom)
data = [" ".join(words) for words in data.get_corpus()]

# # Extract gte_bertopic_embeddings
# model = SentenceTransformer("all-mpnet-base-v2")
# gte_bertopic_embeddings = model.encode(data, show_progress_bar=True)



print(len(data),sent_embeddings_mpnet_aggregated.shape,sent_embeddings_distilroberta_aggregated.shape,sent_embeddings_alibaba_aggregated.shape)

In [ ]:
# default
for i in range(1):
    params = {
        "embedding_model": "all-mpnet-base-v2",
        "nr_topics": [(i+1)*10 for i in range(5)],
        "min_topic_size": 15,
        # "diversity": None,
        "verbose": True
    }
    start_time = datetime.now()
    print("Start_time",start_time)
    
    params_results = []
    
    trainer = Trainer(dataset=dataset,
                      model_name="BERTopic",
                      params=params,
                      bt_embeddings=sent_embeddings_mpnet_aggregated,
                      custom_dataset=custom,
                      verbose=True)
    results,output_mpnet = trainer.train(save=f"../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/sentence/BERTopic_mpnet_{i+1}")
    params_results.append((params, results))
    print("Score:", results[0]["Scores"])
    end_time = datetime.now()
    print("End_time",end_time)
    
    execution_time = end_time - start_time
    print(f" Bertopic-default-all-mpnet-base-v2 Model execution time: {execution_time}")
    with open (f'../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/sentence/Bertopic_mpnet_default_50_TOPICS_{i+1}.pkl', 'wb') as file:
        pickle.dump(output_mpnet,file)
    

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/sentence/time/Bertopic-default-all-mpnet-base-v2_execution_time.txt', 'w') as file:
    file.write(f"Start_time: {start_time} \n end time: {end_time}  Total execution time: {execution_time}\n")

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/sentence/Bertopic-default-all-mpnet-base-v2_params_results.txt', 'w') as file:
    file.write(f"results: { params_results} , Scores: {results}\n")

In [ ]:
import joblib
bertopic_model_mpnet= joblib.load("../../models/Comparative_study_experiment/1955_1970/Sent_embeddings_default_all-mpnet-base-v2_berttopic.pkl")
bertopic_model_mpnet.get_topic_info()

### DistilRoBERTa (all-distilroberta-v1)

In [ ]:
## Default without aggregation

In [ ]:
# Prepare data
dataset, custom = "../../datasets/compare_study_dataset/1955_1970/aggregation/sentence_embedding_data_512", True
data_loader = DataLoader(dataset)
_, timestamps = data_loader.load_docs()
data = data_loader.load_octis(custom)
data = [" ".join(words) for words in data.get_corpus()]

In [ ]:
print(len(data),sent_embeddings_mpnet_aggregated.shape,sent_embeddings_distilroberta_aggregated.shape)

In [ ]:
# default

for i in range(3):
    params = {
        "embedding_model": "all-distilroberta-v1",
        "nr_topics": [(i+1)*10 for i in range(5)],
        "min_topic_size": 15,
        # "diversity": None,
        "verbose": True
    }
    start_time = datetime.now()
    print("Start_time",start_time)
    
    params_results = []
    
    trainer = Trainer(dataset=dataset,
                      model_name="BERTopic",
                      params=params,
                      bt_embeddings=sent_embeddings_distilroberta_aggregated,
                      custom_dataset=custom,
                      verbose=True)
    results,output_distil = trainer.train(save=f"../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/sentence/BERTopic_distilroberta_{i+1}")
    params_results.append((params, results))
    print("Score:", results[0]["Scores"])
    end_time = datetime.now()
    print("End_time",end_time)
    
    execution_time = end_time - start_time
    print(f" Bertopic-default-all_distilroberta Model execution time: {execution_time}")
    with open (f'../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/sentence/Bertopic_distilroberta_default_50_TOPICS_{i+1}.pkl', 'wb') as file:
        pickle.dump(output_distil,file)

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/sentence/Bertopic-default-all_distilroberta_execution_time.txt', 'w') as file:
    file.write(f"Start_time: {start_time} \n end time: {end_time}  Total execution time: {execution_time}\n")

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/sentence/Bertopic-default-all_distilroberta_params_results.txt', 'w') as file:
    file.write(f"results: { params_results} , Scores: {results}\n")

In [ ]:
with open (f'../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/sentence/Bertopic_distilroberta_default_50_TOPICS_{i+1}.pkl', 'wb') as file:
        pickle.dump(output_distil,file)

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/sentence/time/Bertopic-default-all_distilroberta_execution_time.txt', 'w') as file:
    file.write(f"Start_time: {start_time} \n end time: {end_time}  Total execution time: {execution_time}\n")

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/sentence/Bertopic-default-all_distilroberta_params_results.txt', 'w') as file:
    file.write(f"results: { params_results} , Scores: {results}\n")

In [ ]:
import joblib
bertopic_model_distil= joblib.load("../../models/Comparative_study_experiment/1955_1970/Sent_embeddings_default_all-distilroberta-v1_berttopic.pkl")
bertopic_model_distil.get_topic_info()

# GTE (Alibaba-NLP/gte-base-en-v1.5)

In [ ]:
## Default without aggregation

In [ ]:
# Prepare data
dataset, custom = "../../datasets/compare_study_dataset/1955_1970/aggregation/sentence_embedding_data_8192", True
data_loader = DataLoader(dataset)
_, timestamps = data_loader.load_docs()
data = data_loader.load_octis(custom)
data = [" ".join(words) for words in data.get_corpus()]

In [ ]:
# default

model = SentenceTransformer('Alibaba-NLP/gte-base-en-v1.5', trust_remote_code=True)
for i in range(1):
    params = {
        "embedding_model": model ,
        "nr_topics": [(i+1)*10 for i in range(5)],
        "min_topic_size": 15,
        # "diversity": None,
        "verbose": True
    }
    start_time = datetime.now()
    print("Start_time",start_time)
    
    params_results = []
    
    trainer = Trainer(dataset=dataset,
                      model_name="BERTopic",
                      params=params,
                      bt_embeddings=sent_embeddings_alibaba_aggregated,
                      custom_dataset=custom,
                      verbose=True)
    results,output_alibaba = trainer.train(save=f"../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/sentence/BERTopic_alibaba_{i+1}")
    params_results.append((params, results))
    print(results)
    print("Score:", results[0]["Scores"])
    end_time = datetime.now()
    print("End_time",end_time)
    
    execution_time = end_time - start_time
    print(f" Bertopic-default-Alibaba-NLP_gte-base-en-v1.5 Model execution time: {execution_time}")
    with open (f'../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/sentence/Bertopic_alibaba_default_50_TOPICS_{i+1}.pkl', 'wb') as file:
        pickle.dump(output_alibaba,file)

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/sentence/time/Bertopic-default-Alibaba-NLP_gte-base-en-v1.5_execution_time.txt', 'w') as file:
    file.write(f"Start_time: {start_time} \n end time: {end_time}  Total execution time: {execution_time}\n")

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/sentence/Bertopic-default-Alibaba-NLP_gte-base-en-v1.5_params_results.txt', 'w') as file:
    file.write(f"results: { params_results} , Scores: {results}\n")

In [ ]:
import joblib
bertopic_model_alibaba= joblib.load("../../models/Comparative_study_experiment/with_aggregation/data_1955_1970_with_aggregated_Sent_embeddings_alibaba_berttopic_model.pkl")
bertopic_model_alibaba.get_topic_info()

# Jina (jinaai/jina-embeddings-v2-base-en)

In [ ]:
# Prepare data
dataset, custom = "../../datasets/compare_study_dataset/1955_1970/aggregation/sentence_embedding_data_jina", True
data_loader = DataLoader(dataset)
_, timestamps = data_loader.load_docs()
data = data_loader.load_octis(custom)
data = [" ".join(words) for words in data.get_corpus()]

In [ ]:
# default

model = SentenceTransformer('jinaai/jina-embeddings-v2-base-en')
for i in range(1):
    params = {
        "embedding_model": model ,
        "nr_topics": [(i+1)*10 for i in range(5)],
        "min_topic_size": 15,
        # "diversity": None,
        "verbose": True
    }
    start_time = datetime.now()
    print("Start_time",start_time)
    
    params_results = []
    
    trainer = Trainer(dataset=dataset,
                      model_name="BERTopic",
                      params=params,
                      bt_embeddings=sent_embeddings_jina_aggregated,
                      custom_dataset=custom,
                      verbose=True)
    results,output_jina = trainer.train(save=f"../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/sentence/BERTopic_Jina_{i+1}")
    params_results.append((params, results))
    print(results)
    print("Score:", results[0]["Scores"])
    end_time = datetime.now()
    print("End_time",end_time)
    
    execution_time = end_time - start_time
    print(f" Bertopic-default-jina-embeddings-v2-base-en Model execution time: {execution_time}")
    with open (f'../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/sentence/Bertopic_Jina_default_50_TOPICS_{i+1}.pkl', 'wb') as file:
        pickle.dump(output_jina,file)

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/sentence/time/Bertopic-default-jina-embeddings-v2-base-en_execution_time.txt', 'w') as file:
    file.write(f"Start_time: {start_time} \n end time: {end_time}  Total execution time: {execution_time}\n")

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/sentence/Bertopic-default-jina-embeddings-v2-base-en_params_results.txt', 'w') as file:
    file.write(f"results: { params_results} , Scores: {results}\n")

In [ ]:
import joblib
bertopic_model_jina= joblib.load("../../models/Comparative_study_experiment/1955_1970/Bertopic/Sent_embeddings_default_jina-embeddings-v2-base-en_berttopic.pkl")
bertopic_model_jina.get_topic_info()

# Fully preprocessed text (word-level)

In [ ]:
word_embeddings_mpnet_aggregated=np.load("../../datasets/compare_study_dataset/1955_1970/aggregation/word_embeddings_octis_MPnet.npy")
word_embeddings_distilroberta_aggregated=np.load("../../datasets/compare_study_dataset/1955_1970/aggregation/word_embeddings_octis_distilroberta.npy")
word_embeddings_alibaba_aggregated=np.load("../../datasets/compare_study_dataset/1955_1970/aggregation/word_embeddings_octis_Alibaba-NLP_gte-base-en-v1.5.npy")


word_embeddings_jina_aggregated=np.load("../../datasets/compare_study_dataset/1955_1970/aggregation/word_embeddings_octis_jina-embeddings-v2-base-en.npy")

In [ ]:
# Distilroberta

In [ ]:

from sentence_transformers import SentenceTransformer

# Prepare data
dataset, custom = "../../datasets/compare_study_dataset/1955_1970/aggregation/word_embedding_data_512", True
data_loader = DataLoader(dataset)
_, timestamps = data_loader.load_docs()
data = data_loader.load_octis(custom)
data = [" ".join(words) for words in data.get_corpus()]

# # Extract gte_bertopic_embeddings
# model = SentenceTransformer("all-mpnet-base-v2")
# gte_bertopic_embeddings = model.encode(data, show_progress_bar=True)


# print(len(data),word_embeddings_mpnet.shape,word_embeddings_distilroberta.shape)

In [ ]:
# default

for i in range(1):
    params = {
        "embedding_model": "all-distilroberta-v1",
        "nr_topics": [(i+1)*10 for i in range(5)],
        "min_topic_size": 15,
        # "diversity": None,
        "verbose": True
    }
    start_time = datetime.now()
    print("Start_time",start_time)
    
    params_results = []
    
    trainer = Trainer(dataset=dataset,
                      model_name="BERTopic",
                      params=params,
                      bt_embeddings=word_embeddings_distilroberta_aggregated,
                      custom_dataset=custom,
                      verbose=True)
    results_dist,output_dist_word = trainer.train(save=f"../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/BERTopic_WORD_distilroberta_{i+1}")
    params_results.append((params, results_dist))
    print("Score:", results_dist[0]["Scores"])
    end_time = datetime.now()
    print("End_time",end_time)
    
    execution_time = end_time - start_time
    print(f" Bertopic-default-WORD_all_distilroberta Model execution time: {execution_time}")
    with open (f'../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/Bertopic_WORD_Distil_default_50_TOPICS_{i+1}.pkl', 'wb') as file:
        pickle.dump(output_dist_word,file)

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/time/Bertopic-default-WORD_all_distilroberta_execution_time.txt', 'w') as file:
    file.write(f"Start_time: {start_time} \n end time: {end_time}  Total execution time: {execution_time}\n")

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/Bertopic-default-WORD_all_distilroberta_params_results.txt', 'w') as file:
    file.write(f"results: { params_results} , Scores: {results_dist}\n")

In [ ]:
import joblib
bertopic_model_word_dist= joblib.load("../../models/Comparative_study_experiment/1955_1970/word_embeddings_default_all-distilroberta-v1_berttopic.pkl")
bertopic_model_word_dist.get_topic_info()

# Tuned hyperparameters

In [ ]:
results_df_distil_hypertuned=pd.read_pickle(("../results_aggregated/hyperparameter_tuning_words/Words_params_results_Distilroberta.pkl"))

In [ ]:
 # Hyperparameters: {'min_cluster_size': [14], 'min_samples': [10], 'min_topic_size': [12], 'n_components': [2], 'n_neighbors': [15]}, Objective Value: -0.8358076794549347
# best
# {'min_cluster_size': 29,
#  'min_samples': 11,
#  'min_topic_size': 38,
#  'n_components': 2,
#  'n_neighbors': 16}

In [ ]:
# HYPERTUNED

for i in range(1):
    params = {
        "embedding_model": "all-distilroberta-v1",
        "nr_topics": [(i+1)*10 for i in range(5)],
        "min_topic_size": 38,
        "umap_model": UMAP(angular_rp_forest=True, metric='cosine', n_components=2, n_neighbors=16, n_jobs=1, random_state=42, verbose=True),
        "hdbscan_model": HDBSCAN(gen_min_span_tree=True, min_cluster_size=29, min_samples=11, prediction_data=True),
        # "diversity": None,
        "verbose": True
    }
    start_time = datetime.now()
    print("Start_time",start_time)
    
    params_results = []
    
    trainer = Trainer(dataset=dataset,
                      model_name="BERTopic",
                      params=params,
                      bt_embeddings=word_embeddings_distilroberta_aggregated,
                      custom_dataset=custom,
                      verbose=True)
    results_dist_HYPERTUNED,output_dist_word_HYPERTUNED = trainer.train(save=f"../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/BERTopic_WORD_HYPERTUNED_distilroberta_{i+1}")
    params_results.append((params, results_dist_HYPERTUNED))
    print("Score:", results_dist_HYPERTUNED[0]["Scores"])
    end_time = datetime.now()
    print("End_time",end_time)
    
    execution_time = end_time - start_time
    print(f" Bertopic-HYPERTUNED-WORD_all_distilroberta Model execution time: {execution_time}")
    with open (f'../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/Bertopic_WORD_Distil_HYPERTUNED_50_TOPICS_{i+1}.pkl', 'wb') as file:
        pickle.dump(output_dist_word_HYPERTUNED,file)

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/time/Bertopic-HYPERTUNED-WORD_all_distilroberta_execution_time.txt', 'w') as file:
    file.write(f"Start_time: {start_time} \n end time: {end_time}  Total execution time: {execution_time}\n")

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/Bertopic-HYPERTUNED-WORD_all_distilroberta_params_results.txt', 'w') as file:
    file.write(f"results: { params_results} , Scores: {results_dist_HYPERTUNED}\n")

In [ ]:
# MPNET

In [ ]:
from sentence_transformers import SentenceTransformer

# Prepare data
dataset, custom = "../../datasets/compare_study_dataset/1955_1970/aggregation/word_embedding_data_384", True
data_loader = DataLoader(dataset)
_, timestamps = data_loader.load_docs()
data = data_loader.load_octis(custom)
data = [" ".join(words) for words in data.get_corpus()]

# # Extract gte_bertopic_embeddings
# model = SentenceTransformer("all-mpnet-base-v2")
# gte_bertopic_embeddings = model.encode(data, show_progress_bar=True)

In [ ]:
# default

for i in range(3):
    params = {
        "embedding_model": "all-mpnet-base-v2",
        "nr_topics": [(i+1)*10 for i in range(5)],
        "min_topic_size": 15,
        # "diversity": None,
        "verbose": True
    }
    start_time = datetime.now()
    print("Start_time",start_time)
    
    params_results = []
    
    trainer = Trainer(dataset=dataset,
                      model_name="BERTopic",
                      params=params,
                      bt_embeddings=word_embeddings_mpnet_aggregated,
                      custom_dataset=custom,
                      verbose=True)
    results_mpnet,output_word_mpnet = trainer.train(save=f"../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/BERTopic_WORD_mpnet_{i+1}")
    params_results.append((params, results_mpnet))
    print("Score:", results_mpnet[0]["Scores"])
    end_time = datetime.now()
    print("End_time",end_time)
    
    execution_time = end_time - start_time
    print(f" Bertopic-default-WORD_all-mpnet-base-v2 Model execution time: {execution_time}")
    with open (f'../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/Bertopic_WORD_MPNET_default_50_TOPICS_{i+1}.pkl', 'wb') as file:
        pickle.dump(output_word_mpnet,file)

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/time/Bertopic-default-WORD_all-mpnet-base-v2_execution_time.txt', 'w') as file:
    file.write(f"Start_time: {start_time} \n end time: {end_time}  Total execution time: {execution_time}\n")

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/Bertopic-default-WORD_all-mpnet-base-v2_params_results.txt', 'w') as file:
    file.write(f"results: { params_results} , Scores: {results_mpnet}\n")

In [ ]:
import joblib
bertopic_model_word_MPNET= joblib.load("../../models/Comparative_study_experiment/1955_1970/word_embeddings_default_all-mpnet-base-v2_berttopic.pkl")
bertopic_model_word_MPNET.get_topic_info()

In [ ]:
# # hypertuned 

# # BEST
# # {'min_cluster_size': 28,
# #  'min_samples': 48,
# #  'min_topic_size': 12,
# #  'n_components': 8,
# #  'n_neighbors': 8}

In [ ]:
results_df__mpnet_hypertuned =pd.read_pickle("../results_aggregated/hyperparameter_tuning_words/Words_params_results_MPNET.pkl")

In [ ]:
# HYPERTUNED

for i in range(1):
    params = {
        "embedding_model": "all-mpnet-base-v2",
        "nr_topics": [(i+1)*10 for i in range(5)],
        "min_topic_size": 44,
        "umap_model": UMAP(angular_rp_forest=True, metric='cosine', n_components=6, n_neighbors=14, random_state=42, verbose=True),
        "hdbscan_model": HDBSCAN(gen_min_span_tree=True, min_cluster_size=20, min_samples=21, prediction_data=True),
        # "diversity": None,
        # "diversity": None,
        "verbose": True
    }
    start_time = datetime.now()
    print("Start_time",start_time)
    
    params_results = []
    
    trainer = Trainer(dataset=dataset,
                      model_name="BERTopic",
                      params=params,
                      bt_embeddings=word_embeddings_mpnet_aggregated,
                      custom_dataset=custom,
                      verbose=True)
    results_mpnet_HYPERTUNED,output_word_mpnet_HYPERTUNED = trainer.train(save=f"../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/BERTopic_WORD_HYPERTUNED_mpnet_{i+1}")
    params_results.append((params, results_mpnet_HYPERTUNED))
    print("Score:", results_mpnet_HYPERTUNED[0]["Scores"])
    end_time = datetime.now()
    print("End_time",end_time)
    
    execution_time = end_time - start_time
    print(f" Bertopic-HYPERTUNED-WORD_all-mpnet-base-v2 Model execution time: {execution_time}")
    with open (f'../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/Bertopic_WORD_MPNET_HYPERTUNED_50_TOPICS_{i+1}.pkl', 'wb') as file:
        pickle.dump(output_word_mpnet_HYPERTUNED,file)

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/time/Bertopic-HYPERTUNED-WORD_all-mpnet-base-v2_execution_time.txt', 'w') as file:
    file.write(f"Start_time: {start_time} \n end time: {end_time}  Total execution time: {execution_time}\n")

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/Bertopic-HYPERTUNED-WORD_all-mpnet-base-v2_params_results.txt', 'w') as file:
    file.write(f"results: { params_results} , Scores: {results_mpnet_HYPERTUNED}\n")

# GTE (Alibaba-NLP/gte-base-en-v1.5)

In [ ]:
# default

In [ ]:
# Prepare data
dataset, custom = "../../datasets/compare_study_dataset/1955_1970/aggregation/word_embedding_data_8192", True
data_loader = DataLoader(dataset)
_, timestamps = data_loader.load_docs()
data = data_loader.load_octis(custom)
data = [" ".join(words) for words in data.get_corpus()]

In [ ]:
# default

model = SentenceTransformer('Alibaba-NLP/gte-base-en-v1.5', trust_remote_code=True)
for i in range(1):
    params = {
        "embedding_model": model ,
        "nr_topics": [(i+1)*10 for i in range(5)],
        "min_topic_size": 15,
        # "diversity": None,
        "verbose": True
    }
    start_time = datetime.now()
    print("Start_time",start_time)
    
    params_results = []
    
    trainer = Trainer(dataset=dataset,
                      model_name="BERTopic",
                      params=params,
                      bt_embeddings=word_embeddings_alibaba_aggregated,
                      custom_dataset=custom,
                      verbose=True)
    results_alibaba,output_word_alibaba = trainer.train(save=f"../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/BERTopic_WORD_Alibaba_{i+1}")
    params_results.append((params, results_alibaba))
    print(results_alibaba)
    print("Score:", results_alibaba[0]["Scores"])
    end_time = datetime.now()
    print("End_time",end_time)
    
    execution_time = end_time - start_time
    print(f" Bertopic-default-Alibaba-NLP_gte-base-en-v1.5 Model execution time: {execution_time}")
    with open (f'../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/Bertopic_WORD_Alibaba_default_50_TOPICS_{i+1}.pkl', 'wb') as file:
        pickle.dump(output_word_alibaba,file)

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/time/Bertopic-default-WORD_Alibaba-NLP_gte-base-en-v1.5_execution_time.txt', 'w') as file:
    file.write(f"Start_time: {start_time} \n end time: {end_time}  Total execution time: {execution_time}\n")

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/Bertopic-default-WORD_Alibaba-NLP_gte-base-en-v1.5_params_results.txt', 'w') as file:
    file.write(f"results: { params_results} , Scores: {results_alibaba}\n")

In [ ]:
import joblib
bertopic_model_word_Alibaba= joblib.load("../../models/Comparative_study_experiment/1955_1970/word_embeddings_default_alibaba_gte_base_berttopic.pkl")
bertopic_model_word_Alibaba.get_topic_info()

In [ ]:
# Hypertuned

In [ ]:
 # Hyperparameters: {'min_cluster_size': [31], 'min_samples': [24], 'min_topic_size': [37], 'n_components': [5], 'n_neighbors': [10]}, Objective Value: -0.7670762290091766

In [ ]:
# HYPERTUNED

model = SentenceTransformer('Alibaba-NLP/gte-base-en-v1.5', trust_remote_code=True)
for i in range(1):
    params = {
        "embedding_model": model ,
        "nr_topics": [(i+1)*10 for i in range(5)],
        "min_topic_size": 37,
        "umap_model": UMAP(angular_rp_forest=True, metric='cosine', n_components=5, n_neighbors=10, random_state=42, verbose=True),
        "hdbscan_model": HDBSCAN(gen_min_span_tree=True, min_cluster_size=31, min_samples=24, prediction_data=True),
        "verbose": True
    }
    start_time = datetime.now()
    print("Start_time",start_time)
    
    params_results = []
    
    trainer = Trainer(dataset=dataset,
                      model_name="BERTopic",
                      params=params,
                      bt_embeddings=word_embeddings_alibaba_aggregated,
                      custom_dataset=custom,
                      verbose=True)
    results_alibaba_HYPERTUNED,output_word_alibaba_HYPERTUNED = trainer.train(save=f"../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/BERTopic_WORD_HYPERTUNED_Alibaba_{i+1}")
    params_results.append((params, results_alibaba_HYPERTUNED))
    print(results_alibaba_HYPERTUNED)
    print("Score:", results_alibaba_HYPERTUNED[0]["Scores"])
    end_time = datetime.now()
    print("End_time",end_time)
    
    execution_time = end_time - start_time
    print(f" Bertopic-HYPERTUNED-Alibaba-NLP_gte-base-en-v1.5 Model execution time: {execution_time}")
    with open (f'../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/Bertopic_WORD_Alibaba_HYPERTUNED_50_TOPICS_{i+1}.pkl', 'wb') as file:
        pickle.dump(output_word_alibaba_HYPERTUNED,file)

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/time/Bertopic-HYPERTUNED-WORD_Alibaba-NLP_gte-base-en-v1.5_execution_time.txt', 'w') as file:
    file.write(f"Start_time: {start_time} \n end time: {end_time}  Total execution time: {execution_time}\n")

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/Bertopic-HYPERTUNED-WORD_Alibaba-NLP_gte-base-en-v1.5_params_results.txt', 'w') as file:
    file.write(f"results: { params_results} , Scores: {results_alibaba_HYPERTUNED}\n")

# Jina (jinaai/jina-embeddings-v2-base-en)

In [ ]:
# Prepare data
dataset, custom = "../../datasets/compare_study_dataset/1955_1970/aggregation/word_embedding_data_jina", True
data_loader = DataLoader(dataset)
_, timestamps = data_loader.load_docs()
data = data_loader.load_octis(custom)
data = [" ".join(words) for words in data.get_corpus()]

In [ ]:
# default

model = SentenceTransformer('jinaai/jina-embeddings-v2-base-en')
for i in range(1):
    params = {
        "embedding_model": model ,
        "nr_topics": [(i+1)*10 for i in range(5)],
        "min_topic_size": 15,
        # "diversity": None,
        "verbose": True
    }
    start_time = datetime.now()
    print("Start_time",start_time)
    
    params_results = []
    
    trainer = Trainer(dataset=dataset,
                      model_name="BERTopic",
                      params=params,
                      bt_embeddings=word_embeddings_jina_aggregated,
                      custom_dataset=custom,
                      verbose=True)
    results_jina,output_word_jina = trainer.train(save=f"../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/BERTopic_WORD_Jina_{i+1}")
    params_results.append((params, results_jina))
    print(results_jina)
    print("Score:", results_jina[0]["Scores"])
    end_time = datetime.now()
    print("End_time",end_time)
    
    execution_time = end_time - start_time
    print(f" Bertopic-default-jina-embeddings-v2-base-en Model execution time: {execution_time}")
    with open (f'../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/Bertopic_WORD_jina_default_50_TOPICS_{i+1}.pkl', 'wb') as file:
        pickle.dump(output_word_jina,file)

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/time/Bertopic-default-WORD_jina-embeddings-v2-base-en_execution_time.txt', 'w') as file:
    file.write(f"Start_time: {start_time} \n end time: {end_time}  Total execution time: {execution_time}\n")

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/Bertopic-default-WORD_jina-embeddings-v2-base-en_params_results.txt', 'w') as file:
    file.write(f"results: { params_results} , Scores: {results_jina}\n")

In [ ]:
# HYPERTUNED

model = SentenceTransformer('jinaai/jina-embeddings-v2-base-en')
for i in range(1):
    params = {
        "embedding_model": model ,
        "nr_topics": [(i+1)*10 for i in range(5)],
        "min_topic_size": 37,
        "umap_model": UMAP(angular_rp_forest=True, metric='cosine', n_components=5, n_neighbors=10, random_state=42, verbose=True),
        "hdbscan_model": HDBSCAN(gen_min_span_tree=True, min_cluster_size=31, min_samples=24, prediction_data=True),
        "verbose": True
    }
    start_time = datetime.now()
    print("Start_time",start_time)
    
    params_results = []
    
    trainer = Trainer(dataset=dataset,
                      model_name="BERTopic",
                      params=params,
                      bt_embeddings=word_embeddings_jina_aggregated,
                      custom_dataset=custom,
                      verbose=True)
    results_jina_HYPERTUNED,output_word_jina_HYPERTUNED = trainer.train(save=f"../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/BERTopic_WORD_HYPERTUNED_jina_{i+1}")
    params_results.append((params, results_jina_HYPERTUNED))
    print(results_jina_HYPERTUNED)
    print("Score:", results_jina_HYPERTUNED[0]["Scores"])
    end_time = datetime.now()
    print("End_time",end_time)
    
    execution_time = end_time - start_time
    print(f" Bertopic-HYPERTUNED-jina-NLP_gte-base-en-v1.5 Model execution time: {execution_time}")
    with open (f'../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/Bertopic_WORD_jina_HYPERTUNED_50_TOPICS_{i+1}.pkl', 'wb') as file:
        pickle.dump(output_word_jina_HYPERTUNED,file)

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/time/Bertopic-HYPERTUNED-WORD_jina-NLP_gte-base-en-v1.5_execution_time.txt', 'w') as file:
    file.write(f"Start_time: {start_time} \n end time: {end_time}  Total execution time: {execution_time}\n")

with open('../evaluation_results/Results_Comp_Study/1955_1970/Bertopic/word/Bertopic-HYPERTUNED-WORD_jina-NLP_gte-base-en-v1.5_params_results.txt', 'w') as file:
    file.write(f"results: { params_results} , Scores: {output_word_jina_HYPERTUNED}\n")